In [3]:
#!hdfs dfs -mkdir -p /data

In [ ]:
###
!hdfs dfs -put C:\Users\vongo\Final_BigData\flights.csv /data/
!hdfs dfs -put C:\Users\vongo\Final_BigData\airlines.csv /data/
!hdfs dfs -put C:\Users\vongo\Final_BigData\airports.csv /data/
###

In [5]:
!hdfs dfs -ls /data/

Found 3 items
-rw-r--r--   1 vongo supergroup        359 2026-06-12 00:45 /data/airlines.csv
-rw-r--r--   1 vongo supergroup      23867 2026-06-12 00:45 /data/airports.csv
-rw-r--r--   1 vongo supergroup  592406591 2026-06-12 00:45 /data/flights.csv


In [6]:
import os
import shutil

print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
print("java =", shutil.which("java"))
print("spark-submit =", shutil.which("spark-submit"))

JAVA_HOME = C:\java\openjdk-17.0.18b8
java = C:\java\openjdk-17.0.18b8\bin\java.EXE
spark-submit = C:\Users\vongo\Final_BigData\.venv\Scripts\spark-submit.CMD


In [7]:
import pyspark
print("PySpark:", pyspark.__version__)

PySpark: 4.1.1


In [8]:
from pyspark.sql import SparkSession

In [9]:
spark = SparkSession.builder \
    .appName("Flight Data Analysis") \
    .getOrCreate()

In [10]:
flights_df  = spark.read.csv("hdfs://localhost:9000/data/flights.csv",   header=True, inferSchema=True)
airlines_df = spark.read.csv("hdfs://localhost:9000/data/airlines.csv",  header=True, inferSchema=True)
airports_df = spark.read.csv("hdfs://localhost:9000/data/airports.csv",  header=True, inferSchema=True)

flights_df.createOrReplaceTempView("flights")
airlines_df.createOrReplaceTempView("airlines")
airports_df.createOrReplaceTempView("airports")

In [11]:
import pandas as pd

# QUERY 1: Systemic Failure Detection – Airports with the Worst On-Time Departure Rate

In [17]:
query1 = spark.sql("""
    SELECT
        f.ORIGIN_AIRPORT                                                       AS airport_code,
        a.CITY                                                                 AS city,
        a.STATE                                                                AS state,
        COUNT(*)                                                               AS total_flights,
        ROUND(SUM(CASE WHEN f.DEPARTURE_DELAY > 15 THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                                                   AS delay_rate_pct,
        ROUND(AVG(f.DEPARTURE_DELAY), 2)                                       AS avg_dep_delay_min,
        ROUND(AVG(f.AIR_SYSTEM_DELAY), 2)                                      AS avg_air_system_delay,
        ROUND(AVG(f.AIRLINE_DELAY), 2)                                         AS avg_airline_delay,
        ROUND(AVG(f.WEATHER_DELAY), 2)                                         AS avg_weather_delay
    FROM flights f
    LEFT JOIN airports a ON f.ORIGIN_AIRPORT = a.IATA_CODE
    WHERE f.CANCELLED = 0
      AND f.DEPARTURE_DELAY IS NOT NULL
    GROUP BY f.ORIGIN_AIRPORT, a.CITY, a.STATE
    HAVING COUNT(*) >= 1000
    ORDER BY delay_rate_pct DESC
    LIMIT 20
""")
df1 = query1.toPandas()

print("=== Q1: Airports with Worst On-Time Departure Rate (by Delay Source) ===")
df1.head()

=== Q1: Airports with Worst On-Time Departure Rate (by Delay Source) ===


,airport_code,city,state,total_flights,delay_rate_pct,avg_dep_delay_min,avg_air_system_delay,avg_airline_delay,avg_weather_delay
0,ASE,Aspen,CO,3286,27.08,17.46,14.63,16.36,5.33
1,MDW,Chicago,IL,78927,23.84,12.82,8.73,18.30,3.31
2,BWI,Baltimore,MD,84546,23.68,13.31,10.64,21.55,3.29
3,ORD,Chicago,IL,277336,23.59,14.07,14.21,18.91,6.34
4,EWR,Newark,NJ,98662,23.02,13.58,13.16,22.30,2.73


# QUERY 2: Geographical Weather Bottlenecks – States Most Affected by Weather Delay

In [18]:
query2 = spark.sql("""
    SELECT
        a.STATE                                                                     AS state,
        COUNT(*)                                                                    AS total_departures,
        ROUND(SUM(CASE WHEN f.WEATHER_DELAY > 0 THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                                                        AS weather_affected_pct,
        ROUND(AVG(CASE WHEN f.WEATHER_DELAY > 0 THEN f.WEATHER_DELAY END), 2)      AS avg_weather_delay_when_affected,
        SUM(CASE WHEN f.CANCELLED = 1
                  AND f.CANCELLATION_REASON = 'B' THEN 1 ELSE 0 END)               AS weather_cancellations,
        ROUND(SUM(CASE WHEN f.CANCELLED = 1
                        AND f.CANCELLATION_REASON = 'B' THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                                                        AS weather_cancel_rate_pct
    FROM flights f
    LEFT JOIN airports a ON f.ORIGIN_AIRPORT = a.IATA_CODE
    WHERE a.STATE IS NOT NULL
    GROUP BY a.STATE
    HAVING COUNT(*) >= 500
    ORDER BY weather_affected_pct DESC
    LIMIT 20
""")
df2 = query2.toPandas()
print("=== Q2: Geographical Weather Bottlenecks by State ===")
query2.show(truncate=False)
df2.head()

=== Q2: Geographical Weather Bottlenecks by State ===
+-----+----------------+--------------------+-------------------------------+---------------------+-----------------------+
|state|total_departures|weather_affected_pct|avg_weather_delay_when_affected|weather_cancellations|weather_cancel_rate_pct|
+-----+----------------+--------------------+-------------------------------+---------------------+-----------------------+
|IL   |381644          |2.79                |46.39                          |6583                 |1.72                   |
|GA   |360496          |1.92                |45.7                           |1796                 |0.50                   |
|TX   |631124          |1.60                |48.06                          |8466                 |1.34                   |
|IA   |17382           |1.60                |60.46                          |320                  |1.84                   |
|AR   |22654           |1.43                |68.14                          |3

,state,total_departures,weather_affected_pct,avg_weather_delay_when_affected,weather_cancellations,weather_cancel_rate_pct
0,IL,381644,2.79,46.39,6583,1.72
1,GA,360496,1.92,45.70,1796,0.50
2,TX,631124,1.60,48.06,8466,1.34
3,IA,17382,1.60,60.46,320,1.84
4,AR,22654,1.43,68.14,362,1.60


# QUERY 3: High-Frequency, High-Cancellation Flight Paths


In [19]:
query3 = spark.sql("""
    SELECT
        f.ORIGIN_AIRPORT                                                        AS origin,
        ap1.CITY                                                                AS origin_city,
        f.DESTINATION_AIRPORT                                                   AS destination,
        ap2.CITY                                                                AS dest_city,
        COUNT(*)                                                                AS total_flights,
        SUM(f.CANCELLED)                                                        AS total_cancellations,
        ROUND(SUM(f.CANCELLED) * 100.0 / COUNT(*), 2)                          AS cancellation_rate_pct,
        SUM(CASE WHEN f.CANCELLATION_REASON = 'A' THEN 1 ELSE 0 END)           AS cancelled_by_airline,
        SUM(CASE WHEN f.CANCELLATION_REASON = 'B' THEN 1 ELSE 0 END)           AS cancelled_by_weather,
        SUM(CASE WHEN f.CANCELLATION_REASON = 'C' THEN 1 ELSE 0 END)           AS cancelled_by_nas,
        SUM(CASE WHEN f.CANCELLATION_REASON = 'D' THEN 1 ELSE 0 END)           AS cancelled_by_security
    FROM flights f
    LEFT JOIN airports ap1 ON f.ORIGIN_AIRPORT      = ap1.IATA_CODE
    LEFT JOIN airports ap2 ON f.DESTINATION_AIRPORT = ap2.IATA_CODE
    GROUP BY f.ORIGIN_AIRPORT, ap1.CITY, f.DESTINATION_AIRPORT, ap2.CITY
    HAVING COUNT(*) >= 500                        -- chỉ xét route tần suất cao
       AND SUM(f.CANCELLED) * 100.0 / COUNT(*) >= 2.0  -- tỷ lệ hủy >= 2%
    ORDER BY cancellation_rate_pct DESC, total_flights DESC
    LIMIT 20
""")
df3 = query3.toPandas()
print("=== Q3: High-Frequency, High-Cancellation Flight Paths ===")
df3.head()

=== Q3: High-Frequency, High-Cancellation Flight Paths ===


,origin,origin_city,destination,dest_city,total_flights,total_cancellations,cancellation_rate_pct,cancelled_by_airline,cancelled_by_weather,cancelled_by_nas,cancelled_by_security
0,LGA,New York,ORF,Norfolk,515,68,13.20,26,23,19,0
1,DCA,Arlington,JFK,New York,996,114,11.45,34,55,25,0
2,JFK,New York,DCA,Arlington,1002,106,10.58,29,55,22,0
3,RDU,Raleigh,LGA,New York,1878,197,10.49,58,79,60,0
4,LGA,New York,GSO,Greensboro,1295,134,10.35,56,41,37,0


# QUERY 4: Cascading Delay Analysis – Identifying Airlines with Late Aircraft Problem

In [21]:
query4 = spark.sql("""
    SELECT
        al.AIRLINE                                                                          AS airline_name,
        COUNT(*)                                                                            AS total_delayed_flights,
        ROUND(AVG(f.DEPARTURE_DELAY), 2)                                                    AS avg_total_dep_delay,
        ROUND(AVG(f.LATE_AIRCRAFT_DELAY), 2)                                                AS avg_late_aircraft_delay,
        ROUND(AVG(f.AIRLINE_DELAY), 2)                                                      AS avg_airline_delay,
        ROUND(AVG(f.AIR_SYSTEM_DELAY), 2)                                                   AS avg_air_system_delay,
        ROUND(AVG(f.WEATHER_DELAY), 2)                                                      AS avg_weather_delay,
        ROUND(
            AVG(f.LATE_AIRCRAFT_DELAY) * 100.0 /
            NULLIF(AVG(f.AIR_SYSTEM_DELAY) + AVG(f.AIRLINE_DELAY) +
                   AVG(f.LATE_AIRCRAFT_DELAY) + AVG(f.WEATHER_DELAY) +
                   AVG(f.SECURITY_DELAY), 0),
        2)                                                                                  AS late_aircraft_share_pct
    FROM flights f
    JOIN airlines al ON f.AIRLINE = al.IATA_CODE
    WHERE f.CANCELLED = 0
      AND f.DEPARTURE_DELAY > 15
    GROUP BY al.AIRLINE
    ORDER BY late_aircraft_share_pct DESC
""")
df4 = query4.toPandas()
print("=== Q4: Cascading Delay Analysis (Late Aircraft Share per Airline) ===")
df4.head()

=== Q4: Cascading Delay Analysis (Late Aircraft Share per Airline) ===


,airline_name,total_delayed_flights,avg_total_dep_delay,avg_late_aircraft_delay,avg_airline_delay,avg_air_system_delay,avg_weather_delay,late_aircraft_share_pct
0,Southwest Airlines Co.,253959,52.39,30.98,18.50,5.04,2.69,54.10
1,Skywest Airlines Inc.,93380,66.35,34.77,24.78,9.36,3.04,48.25
2,Frontier Airlines Inc.,20127,72.06,35.67,18.99,24.79,1.19,44.23
3,Hawaiian Airlines Inc.,5230,48.65,23.38,28.17,0.50,1.30,43.77
4,United Air Lines Inc.,115902,64.59,31.87,25.22,12.40,3.74,43.51


# QUERY 5: Monthly Delay Seasonality – Detecting Systemic Seasonal Bottlenecks


In [22]:
query5 = spark.sql("""
    SELECT
        MONTH,
        COUNT(*)                                                                AS total_flights,
        ROUND(SUM(CANCELLED) * 100.0 / COUNT(*), 2)                            AS cancellation_rate_pct,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN DEPARTURE_DELAY END), 2)        AS avg_dep_delay,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN WEATHER_DELAY END), 2)          AS avg_weather_delay,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN AIR_SYSTEM_DELAY END), 2)       AS avg_air_system_delay,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN AIRLINE_DELAY END), 2)          AS avg_airline_delay,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN LATE_AIRCRAFT_DELAY END), 2)    AS avg_late_aircraft_delay
    FROM flights
    GROUP BY MONTH
    ORDER BY MONTH
""")
df5 = query5.toPandas()
print("=== Q5: Monthly Delay Seasonality – Systemic Seasonal Bottlenecks ===")
df5.head()

=== Q5: Monthly Delay Seasonality – Systemic Seasonal Bottlenecks ===


,MONTH,total_flights,cancellation_rate_pct,avg_dep_delay,avg_weather_delay,avg_air_system_delay,avg_airline_delay,avg_late_aircraft_delay
0,1,469968,2.55,9.73,2.74,13.32,17.80,22.76
1,2,429191,4.78,11.81,4.32,14.18,17.99,22.67
2,3,504312,2.18,9.63,2.40,12.87,19.05,22.59
3,4,485151,0.93,7.68,2.69,13.59,18.12,21.71
4,5,496993,1.15,9.42,3.75,14.00,18.61,24.23
